# 06 · Del reportero al dataset · la regla de oro

**Parte III · El oficio.** En el notebook 05 viste que Qwen con notas ya no inventa tanto, pero **no sabe el oficio**: no da 5 bullets, no cita bien, a veces añade una frase de su cosecha. El arreglo es enseñarle con ejemplos (SFT + LoRA). Este notebook es la mitad que nadie quiere hacer y que decide todo: **los datos**.

Sin torch, sin GPU. Lo que haces aquí:

1. Bajas notas reales con el reportero (`rss_reportero.py`).
2. Las conviertes en filas de chat `system / user / assistant`.
3. Pasas cada fila por el juez (`don_titular.py`) y **tiras** las que inventan.
4. Ves exactamente qué parte de la fila entrena (la máscara del SFT).
5. Separas un hold-out que no haga trampa.

## Dummy (antes de la fórmula)

El modelo es un copiador muy bueno. Si en 200 ejemplos el briefing siempre tiene 5 bullets y cita el medio, va a hacer eso. Si en **uno** de esos ejemplos el assistant menciona un terremoto que no está en las notas, también aprende eso: "a veces se vale agregar cosas". No hay sermón en el system prompt que le gane a un ejemplo.

Por eso la regla de oro es mecánica: **si el assistant dice algo que no está en el user, la fila se tira**. No se corrige "un poquito": se tira y se cuenta.

## Matemática (la misma idea, en serio)

Una fila es $(s, q, d, y^\star)$: system, pregunta, notas, respuesta canónica. Sea $\mathcal{H}(y)$ el conjunto de hechos atómicos de $y$. La regla de oro pide

$$
\forall h \in \mathcal{H}(y^\star):\quad d \models h \qquad(\text{cada hecho tiene soporte en las notas}).
$$

SFT minimiza la misma entropía cruzada del lab 0, pero solo en los tokens del assistant. Con $m_t = 1$ si el token $t$ es del assistant:

$$
\mathcal{L}_{\mathrm{SFT}}(\theta) = -\frac{1}{\sum_t m_t}\sum_t m_t \log P_\theta(x_t \mid x_{<t}).
$$

Si $y^\star$ contiene un hecho $h$ sin soporte, el gradiente **sube** $P_\theta(h \mid s,q,d)$: le estás pagando al modelo por inventar. Un 2% de filas malas es un 2% del gradiente empujando al villano.

## 0 · Cargar el reportero y el juez

In [1]:
# Colab: descomenta (una vez). No necesita torch.
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/rss_reportero.py
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/don_titular.py
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/ejemplo-briefing.jsonl

import json
import re
import sys
from datetime import date
from pathlib import Path

OFICIO = next((p for p in (Path("."), Path("../parte-3-oficio")) if (p / "don_titular.py").exists()), Path("."))
sys.path.insert(0, str(OFICIO.resolve()))
import don_titular as dt
import rss_reportero as rss

print("system de Don Titular:\n ", dt.SYSTEM)

system de Don Titular:
  Eres Don Titular. Resumes noticias en exactamente 5 bullets. Cita el medio entre paréntesis. No inventes hechos que no estén en las notas. Si un tema no aparece, di 'no aparece'.


## 1 · El reportero: notas de hoy, cero IA

Intenta bajar el feed de BBC Mundo. Si no hay red (o el feed cambió), usa una muestra guardada. **Las dos cosas son normales**: un sistema serio documenta qué pasa cuando la fuente falla.

In [2]:
FEED, MEDIO = "https://feeds.bbci.co.uk/mundo/rss.xml", "BBC Mundo"

MUESTRA_XML = b"""<?xml version="1.0"?><rss><channel>
<item><title>Banxico mantiene la tasa de referencia</title><description>El banco central dejo sin cambio el objetivo.</description><link>https://ejemplo/1</link></item>
<item><title>Ola de calor en el norte de Mexico</title><description>Varios estados superan los 40 grados.</description><link>https://ejemplo/2</link></item>
<item><title>La UNAM publica su calendario escolar</title><description>Las inscripciones abren en agosto.</description><link>https://ejemplo/3</link></item>
</channel></rss>"""

try:
    items = rss.parse_items(rss.fetch_feed(FEED))
    origen = f"en vivo · {FEED}"
except Exception as exc:
    items = rss.parse_items(MUESTRA_XML)
    origen = f"muestra offline ({type(exc).__name__})"

print(f"{len(items)} items · {origen}\n")
print(rss.format_notes(items, None, 3))

34 items · en vivo · https://feeds.bbci.co.uk/mundo/rss.xml

Notas del 2026-10-02:
1. "Tenemos miedo a perder la vida aquí": los migrantes de América Latina que EE.UU. deporta a República Centroafricana
   Tenían protecciones ante la deportación ordenadas por un tribunal de inmigración y acabaron en otro continente, en un país del que no tenían referencias. Allí esperan un "milagro" para poder regresar a EE.UU.
   fuente: https://www.bbc.com/mundo/articles/c8046w1kgpgwo?at_medium=RSS&at_campaign=rss
2. El Parlamento de España vota en contra de los dos decretos sobre vivienda del gobierno de Pedro Sánchez
   La votación se dio en medio de protestas por la situación de la vivienda en en país europeo.
   fuente: https://www.bbc.com/mundo/articles/cqj9klj2zy29o?at_medium=RSS&at_campaign=rss
3. "He presenciado casi 500 ejecuciones, pero ninguna como la de Christa Pike"
   El periodista de Texas, Michael Graczyk ha sido testigo de varias ejecuciones fallidas a lo largo de su carrera, pero ni

Ese bloque crudo no dice **qué medio** publicó cada nota (todo viene del mismo feed). El juez necesita el medio para revisar las citas, así que lo agregamos con `notas_desde_items`: mismo formato que el dataset (`N. Medio — Título`).

In [3]:
HOY = date.today().isoformat()
NOTAS_HOY = dt.notas_desde_items(items, MEDIO, HOY, limite=3)
print(NOTAS_HOY)

Notas del 2026-10-02:
1. BBC Mundo — "Tenemos miedo a perder la vida aquí": los migrantes de América Latina que EE.UU. deporta a República Centroafricana
   Tenían protecciones ante la deportación ordenadas por un tribunal de inmigración y acabaron en otro continente, en un país del que no tenían referencias. Allí esperan un "milagro" para poder regresar a EE.UU.
2. BBC Mundo — El Parlamento de España vota en contra de los dos decretos sobre vivienda del gobierno de Pedro Sánchez
   La votación se dio en medio de protestas por la situación de la vivienda en en país europeo.
3. BBC Mundo — "He presenciado casi 500 ejecuciones, pero ninguna como la de Christa Pike"
   El periodista de Texas, Michael Graczyk ha sido testigo de varias ejecuciones fallidas a lo largo de su carrera, pero ninguna en la que un condenado haya sobrevivido.


## 2 · Una fila de chat

Una fila del JSONL es una obra de un acto con tres papeles. Para que el SFT tenga qué copiar, **tú** escribes el assistant. Un borrador mecánico ayuda a no empezar de cero, pero no es el briefing: copia títulos y deja el tema vacío.

In [4]:
def borrador(notas: str) -> str:
    """Esqueleto del assistant: un bullet por nota y 'no aparece' hasta llegar a 5. Edítalo a mano."""
    lineas = [f"{i}) <tema> ({n.medio}): {n.titulo.rstrip('.')}." for i, n in enumerate(dt.parse_notas(notas)[:dt.N_BULLETS], 1)]
    lineas += [f"{i}) <tema>: no aparece." for i in range(len(lineas) + 1, dt.N_BULLETS + 1)]
    return "\n".join(lineas)


print(borrador(NOTAS_HOY))
print("\n→ juez:", dt.juzgar(NOTAS_HOY, borrador(NOTAS_HOY)).resumen())

1) <tema> (BBC Mundo): "Tenemos miedo a perder la vida aquí": los migrantes de América Latina que EE.UU. deporta a República Centroafricana.
2) <tema> (BBC Mundo): El Parlamento de España vota en contra de los dos decretos sobre vivienda del gobierno de Pedro Sánchez.
3) <tema> (BBC Mundo): "He presenciado casi 500 ejecuciones, pero ninguna como la de Christa Pike".
4) <tema>: no aparece.
5) <tema>: no aparece.

→ juez: formato ✔ · citas ✔ · soporte 100% · 0 problema(s)


El borrador **pasa el juez**. Eso dice más del juez que del borrador: el juez revisa forma, citas y soporte; no revisa si el bullet *resume* o si el tema tiene sentido. Un script de 5 renglones no es un noticiero. La calidad la pones tú al editar.

## 3 · El dataset de ejemplo pasa la regla de oro

In [5]:
DATASET = OFICIO / "ejemplo-briefing.jsonl"
total, tirados = dt.validar_jsonl(DATASET)
print(f"{total - len(tirados)}/{total} filas pasan")

fila = json.loads(DATASET.read_text(encoding="utf-8").splitlines()[0])
for m in fila["messages"]:
    print(f"\n[{m['role']}]\n{m['content']}")

44/44 filas pasan

[system]
Eres Don Titular. Resumes noticias en exactamente 5 bullets. Cita el medio entre paréntesis. No inventes hechos que no estén en las notas. Si un tema no aparece, di 'no aparece'.

[user]
Notas del 2026-08-14:
1. El País — Banxico mantiene la tasa de referencia.
   El banco central dejó sin cambio el objetivo ante la inflación subyacente.
2. Local — Paro parcial en la L12 del Metro por fallas eléctricas.
   Autoridades estiman reanudar el servicio en la tarde.
3. BBC Mundo — Ola de calor en el norte de México.
   Varios estados superan los 40 °C.

[assistant]
1) Banxico (El País): mantiene la tasa; menciona inflación subyacente.
2) Metro L12 (local): paro parcial por fallas eléctricas; reanudación estimada por la tarde.
3) Clima (BBC Mundo): ola de calor en el norte, varios estados >40 °C.
4) Política nacional: no aparece.
5) Deportes / mercados: no aparece.


## 4 · Filas que se tiran

Mismas notas del 14 de agosto, cinco assistants "casi bien". El juez decide.

In [6]:
NOTAS = fila["messages"][1]["content"]
BUENO = fila["messages"][2]["content"]

candidatas = {
    "canónica": BUENO,
    "terremoto inventado": BUENO.replace("4) Política nacional: no aparece.", "4) Sismo (Local): terremoto en Oaxaca sin daños."),
    "número cambiado": BUENO.replace(">40 °C", ">45 °C"),
    "medio equivocado": BUENO.replace("(BBC Mundo)", "(Reforma)"),
    "solo 4 bullets": "\n".join(BUENO.splitlines()[:4]),
    "dice lo contrario": BUENO.replace("mantiene la tasa", "baja la tasa"),
}
for nombre, assistant in candidatas.items():
    j = dt.juzgar(NOTAS, assistant)
    print(f"{'QUEDA' if j.aprobado else 'TIRAR':5} · {nombre:20} · {j.resumen()}")
    for p in j.problemas:
        print(f"        - {p}")

QUEDA · canónica             · formato ✔ · citas ✔ · soporte 100% · 0 problema(s)
TIRAR · terremoto inventado  · formato ✔ · citas ✔ · soporte 75% · 1 problema(s)
        - bullet 4: soporte 0% < 50% — ¿de dónde salió?
TIRAR · número cambiado      · formato ✔ · citas ✔ · soporte 67% · 1 problema(s)
        - bullet 3: números que no están en las notas ['45']
TIRAR · medio equivocado     · formato ✔ · citas ✘ · soporte 100% · 1 problema(s)
        - bullet 3: cita 'Reforma', que no está en las notas
TIRAR · solo 4 bullets       · formato ✘ · citas ✔ · soporte 100% · 1 problema(s)
        - formato: se esperaban bullets 1..5, llegaron [1, 2, 3, 4]
QUEDA · dice lo contrario    · formato ✔ · citas ✔ · soporte 100% · 0 problema(s)


Cuatro de cinco malas se van solas. **"Dice lo contrario" pasa**: "baja la tasa" usa las mismas palabras que las notas ("tasa") y el juez solo cuenta palabras. Esa es la frontera de una métrica barata. Por eso la regla es *revisadas a mano*: el juez filtra lo obvio en 200 filas para que tus ojos se gasten en lo sutil.

## 5 · ¿Qué parte de la fila entrena? (la máscara $m_t$)

El modelo ve la fila completa con el chat template (el `<|im_start|>` del notebook 05). La pérdida solo cuenta lo que está marcado con `▶`. Lo medimos en caracteres para no depender de un tokenizer; con tokens la proporción es parecida.

In [7]:
def con_mascara(mensajes):
    """Lista de (texto, m) con m=1 solo en el contenido del assistant y su cierre."""
    partes = []
    for msg in mensajes:
        partes.append((f"<|im_start|>{msg['role']}\n", 0))
        m = 1 if msg["role"] == "assistant" else 0
        partes.append((msg["content"] + "<|im_end|>", m))
        partes.append(("\n", 0))
    return partes


partes = con_mascara(fila["messages"])
for texto, m in partes:
    for linea in texto.splitlines():
        print(f"{'▶' if m else ' '} {linea}")

n_total = sum(len(t) for t, _ in partes)
n_perdida = sum(len(t) for t, m in partes if m)
print(f"\nm_t = 1 en {n_perdida}/{n_total} caracteres ({n_perdida / n_total:.0%}). El resto es contexto: se lee, no se practica.")

  <|im_start|>system
  Eres Don Titular. Resumes noticias en exactamente 5 bullets. Cita el medio entre paréntesis. No inventes hechos que no estén en las notas. Si un tema no aparece, di 'no aparece'.<|im_end|>
  
  <|im_start|>user
  Notas del 2026-08-14:
  1. El País — Banxico mantiene la tasa de referencia.
     El banco central dejó sin cambio el objetivo ante la inflación subyacente.
  2. Local — Paro parcial en la L12 del Metro por fallas eléctricas.
     Autoridades estiman reanudar el servicio en la tarde.
  3. BBC Mundo — Ola de calor en el norte de México.
     Varios estados superan los 40 °C.<|im_end|>
  
  <|im_start|>assistant
▶ 1) Banxico (El País): mantiene la tasa; menciona inflación subyacente.
▶ 2) Metro L12 (local): paro parcial por fallas eléctricas; reanudación estimada por la tarde.
▶ 3) Clima (BBC Mundo): ola de calor en el norte, varios estados >40 °C.
▶ 4) Política nacional: no aparece.
▶ 5) Deportes / mercados: no aparece.<|im_end|>
  

m_t = 1 en 314/937 ca

Más de la mitad de la fila es contexto que el modelo **no** practica a escribir. Eso es lo que quieres: no le enseñas a escribir notas (las trae el reportero); le enseñas a escribir **el briefing dadas las notas**.

## 6 · Tu dataset: agregar solo lo que pasa

`agregar` es la única puerta al JSONL: valida y, si no pasa, no escribe. La demo usa un archivo `mi-dataset-demo.jsonl` que se reescribe en cada corrida (idempotente).

In [8]:
def agregar(path: Path, notas: str, briefing: str) -> list[str]:
    problemas = dt.validar_ejemplo({"messages": dt.mensajes(notas, briefing)})
    if not problemas:
        with path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps({"messages": dt.mensajes(notas, briefing)}, ensure_ascii=False) + "\n")
    return problemas


DEMO = Path("mi-dataset-demo.jsonl")
DEMO.write_text("", encoding="utf-8")

conteo = {"quedan": 0, "tiradas": 0}
for assistant in candidatas.values():
    problemas = agregar(DEMO, NOTAS, assistant)
    conteo["tiradas" if problemas else "quedan"] += 1

print(conteo, "→", sum(1 for _ in DEMO.open(encoding="utf-8")), "filas en", DEMO)

{'quedan': 2, 'tiradas': 4} → 2 filas en mi-dataset-demo.jsonl


La fila "dice lo contrario" entró. En tu dataset real, esa es la que tú tienes que cazar leyendo.

## 7 · Hold-out sin trampa: separar por fecha

Si el mismo día aparece en train y en hold-out, el modelo ya vio esas notas: el "después" se ve mejor de lo que es. Separa por **fecha**, no al azar.

In [9]:
def fecha_de(fila: dict) -> str:
    user = next(m["content"] for m in fila["messages"] if m["role"] == "user")
    hallada = re.search(r"\d{4}-\d{2}-\d{2}", user)
    return hallada.group(0) if hallada else "sin-fecha"


def split_por_fecha(filas: list[dict], corte: str) -> tuple[list[dict], list[dict]]:
    """train: fechas < corte · holdout: fechas ≥ corte."""
    train = [f for f in filas if fecha_de(f) < corte]
    return train, [f for f in filas if fecha_de(f) >= corte]


filas = [json.loads(linea) for linea in DATASET.read_text(encoding="utf-8").splitlines()]
train, val = split_por_fecha(filas, "2026-08-14")


def rango(fs: list[dict]) -> str:
    fechas = sorted(fecha_de(f) for f in fs)
    return f"{len(fs):2} filas · {fechas[0]} … {fechas[-1]}" if fechas else " 0 filas"


print("train   ", rango(train))
print("val     ", rango(val))

HOLDOUT = [json.loads(linea) for linea in (OFICIO / "holdout.jsonl").read_text(encoding="utf-8").splitlines()]
fechas_holdout = sorted(re.search(r"\d{4}-\d{2}-\d{2}", c["notas"]).group(0) for c in HOLDOUT)
print("hold-out", f"{len(HOLDOUT):2} casos · {fechas_holdout[0]} … {fechas_holdout[-1]}")
assert max(fecha_de(f) for f in filas) < fechas_holdout[0], "fuga: hay filas de entrenamiento del mismo día o después del hold-out"
print("sin fuga: todo el hold-out es posterior a la última fila del dataset")

train    38 filas · 2026-07-01 … 2026-08-13
val       6 filas · 2026-08-14 … 2026-08-19
hold-out 10 casos · 2026-08-20 … 2026-08-29
sin fuga: todo el hold-out es posterior a la última fila del dataset


Dos separaciones distintas, para dos preguntas distintas:

- **val** (dentro del dataset): `sft_lora_noticias.py` aparta un pedazo **al azar** para la curva de val loss. Responde "¿está memorizando?".
- **hold-out** (`holdout.jsonl`, días posteriores): no entra al entrenamiento nunca. Responde "¿sirve en días que no vio?". Es el que usa el notebook 08.

---

## Entregable (sesión 6)

1. Un `mi-dataset.jsonl` con **≥ 20** filas de tu oficio, todas por `agregar` o validadas con `python don_titular.py mi-dataset.jsonl`.
2. El conteo: cuántas escribiste, cuántas tiró el juez, cuántas tiraste tú a mano y por qué (3 ejemplos).
3. El corte de fecha de tu hold-out y por qué no hay fuga.

Con eso corres el gym:

```bash
python sft_lora_noticias.py --data mi-dataset.jsonl --4bit   # Colab T4
```

## Para romperlo

- Escribe una fila que engañe al juez **sin** usar números ni medios falsos. ¿Qué regla le agregarías a `juzgar`?
- Cambia `UMBRAL_SOPORTE` a 0.8. ¿Cuántas filas buenas tira? Ese es el costo de un juez estricto.
- Mide qué fracción de tu dataset tiene "no aparece". Si es 0%, el modelo nunca practica a decir "no sé".